# Poor Man's Lakehouse — Demo Notebook

This notebook demonstrates the key features of the project after the **catalog-agnostic refactor**. The project now has three main entry points:

1. **`get_catalog()`** — PyIceberg catalog factory (single source of truth)
2. **`LakehouseConnection`** — unified lightweight connector (browsing, scans, DuckDB, Ibis, SQL, writes)
3. **`SparkBuilder`** — catalog-specific PySpark session builders

Plus **Sail** — a Rust-powered Spark Connect engine (no JVM).

## Prerequisites

Start the Lakekeeper catalog before running this notebook:

```bash
just up lakekeeper
```

Make sure your `.env` has `CATALOG=lakekeeper`.

For Sail S3 access, set AWS env vars before starting the kernel:

```bash
export AWS_ACCESS_KEY_ID=minioadmin
export AWS_SECRET_ACCESS_KEY=miniopassword
export AWS_ENDPOINT_URL=http://localhost:9000
export AWS_DEFAULT_REGION=eu-central-1
export AWS_ALLOW_HTTP=true
```

---
## 1. Catalog Factory — `get_catalog()`

The `get_catalog()` function is the single source of truth for creating PyIceberg catalog instances.
It supports **Lakekeeper**, **Nessie**, **PostgreSQL**, and **AWS Glue** backends. Every other
module in the project composes it.

In [26]:
from poor_man_lakehouse import get_catalog

# Uses settings.CATALOG (lakekeeper) by default
catalog = get_catalog()

try:
    catalog.create_namespace("default")
except Exception as e:
    print(f"Namespace 'default' already exists: {e}")
# Full PyIceberg API available
print(f"Namespaces: {catalog.list_namespaces()}")
print(f"Tables in 'default': {catalog.list_tables('default')}")

Namespace 'default' already exists: AlreadyExistsException: Namespace name 'default' already exist in warehouse '65df7fe6-2e15-11f1-ad2a-af56441f9d87'
Namespaces: [('default',)]
Tables in 'default': [('default', 'demo_users')]


---
## 2. LakehouseConnection — Catalog Browsing

`LakehouseConnection` is the unified lightweight connector. It replaces the old
`CatalogBrowser`, `PyIcebergClient`, `IbisConnection`, and `PolarsClient` with a single class.

All catalog browsing is JVM-free — powered by PyIceberg under the hood.

In [27]:
from poor_man_lakehouse import LakehouseConnection

conn = LakehouseConnection()
print(conn)

# Browse namespaces and tables
print(f"Namespaces: {conn.list_namespaces()}")
print(f"Tables in 'default': {conn.list_tables('default')}")

LakehouseConnection(catalog_type='lakekeeper')
Namespaces: ['default']
Tables in 'default': ['demo_users']


In [28]:
# If tables exist, inspect schema and snapshot history
tables = conn.list_tables("default")
if tables:
    table_name = tables[0]
    print(f"--- Schema of '{table_name}' ---")
    for col in conn.table_schema("default", table_name):
        print(f"  {col['name']:20s} {col['type']:15s} required={col['required']}")

    print(f"\n--- Snapshot History of '{table_name}' ---")
    for snap in conn.snapshot_history("default", table_name):
        print(f"  ID: {snap['snapshot_id']}  ts: {snap['timestamp_ms']}  op: {snap['summary'].get('operation', 'N/A')}")
else:
    print("No tables yet. We'll create one in section 3.")

--- Schema of 'demo_users' ---
  id                   int             required=False
  name                 string          required=False
  age                  int             required=False

--- Snapshot History of 'demo_users' ---
  ID: 15506493908010213  ts: 1775081046265  op: append
  ID: 1411506824872411312  ts: 1775081188868  op: append
  ID: 3453482025263907408  ts: 1775081041513  op: append
  ID: 5631240523177872133  ts: 1775081646189  op: append
  ID: 8349136903337623594  ts: 1775081519464  op: delete
  ID: 657418986792699751  ts: 1775081188779  op: delete
  ID: 4393031258654458427  ts: 1775081646114  op: delete
  ID: 3350031235523045554  ts: 1775081611127  op: append
  ID: 4994232313922407804  ts: 1775081519542  op: append


---
## 3. DuckDB Engine — SQL & Iceberg Writes

`LakehouseConnection` provides a DuckDB connection with the Iceberg catalog already attached.
DuckDB 1.5+ supports creating, reading, and writing Iceberg tables through REST catalogs.

In [29]:
# Create a table and write data via DuckDB
conn.create_table("default", "demo_users", "id INTEGER, name VARCHAR, age INTEGER")
print("Table created!")

# Insert rows via SQL
conn.write_table(
    "default",
    "demo_users",
    query="SELECT 1 AS id, 'Alice' AS name, 30 AS age UNION ALL SELECT 2, 'Bob', 25 UNION ALL SELECT 3, 'Charlie', 35",
    mode="overwrite",
)
print("Data written!")

2026-04-02 00:14:11.562 | INFO     | poor_man_lakehouse.lakehouse:create_table:354 - Created table lakekeeper.default.demo_users


Table created!


2026-04-02 00:14:11.774 | INFO     | poor_man_lakehouse.lakehouse:write_table:341 - Wrote to lakekeeper.default.demo_users (mode=overwrite) via DuckDB


Data written!


In [30]:
# Query the table via DuckDB SQL
result = conn.sql("SELECT * FROM lakekeeper.default.demo_users ORDER BY id")
print("--- DuckDB SQL ---")
result.execute()

--- DuckDB SQL ---


,id,name,age
0,1,Alice,30
1,2,Bob,25
2,3,Charlie,35


---
## 4. Native Scans — Polars & Arrow (No JVM)

Scan Iceberg tables directly into Polars LazyFrames or PyArrow Tables.
This uses PyIceberg under the hood — no DuckDB, no JVM.

In [31]:
import polars as pl

# Scan to Polars LazyFrame (lazy evaluation)
lf = conn.scan_polars("default", "demo_users")
print(f"LazyFrame schema: {lf.collect_schema()}")

# Filter and collect
result = lf.filter(pl.col("age") > 25).sort("name")
display(result.collect())

LazyFrame schema: Schema({'id': Int32, 'name': String, 'age': Int32})



thread 'async-executor-4' (12056660) panicked at crates/polars-utils/src/row_counter.rs:71:25:
called `Result::unwrap()` on an `Err` value: ComputeError(ErrString("RowCounter: Invalid state: number of rows removed by deletion files (3) is greater than the number of rows physically present (1)"))


PanicException: called `Result::unwrap()` on an `Err` value: ComputeError(ErrString("RowCounter: Invalid state: number of rows removed by deletion files (3) is greater than the number of rows physically present (1)"))

In [ ]:
# Scan to PyArrow Table
arrow_table = conn.scan_arrow("default", "demo_users")
print(f"Arrow schema: {arrow_table.schema}")
print(f"Rows: {arrow_table.num_rows}")
arrow_table.to_pandas()

Arrow schema: id: int32
name: string
age: int32
Rows: 3


,id,name,age
0,1,Alice,30
1,2,Bob,25
2,3,Charlie,35


---
## 5. Table Metadata — Schema & Snapshot History

Now that `demo_users` has data, let's inspect its metadata.

In [ ]:
# Schema inspection
print("--- Schema ---")
for col in conn.table_schema("default", "demo_users"):
    print(f"  {col['name']:15s} {col['type']:10s} required={col['required']}")

# Snapshot history
print("\n--- Snapshot History ---")
for snap in conn.snapshot_history("default", "demo_users"):
    op = snap["summary"].get("operation", "N/A")
    added = snap["summary"].get("added-records", "0")
    print(f"  Snapshot {snap['snapshot_id']}  op={op}  added_records={added}")

# Load the raw PyIceberg Table object for advanced operations
table = conn.load_table("default", "demo_users")
print(f"\nPartition spec: {table.metadata.partition_specs}")
print(f"Current schema ID: {table.metadata.current_schema_id}")

--- Schema ---
  id              int        required=False
  name            string     required=False
  age             int        required=False

--- Snapshot History ---
  Snapshot 1411506824872411312  op=append  added_records=3
  Snapshot 15506493908010213  op=append  added_records=3
  Snapshot 657418986792699751  op=delete  added_records=6
  Snapshot 8349136903337623594  op=delete  added_records=3
  Snapshot 3453482025263907408  op=append  added_records=3
  Snapshot 4994232313922407804  op=append  added_records=3

Partition spec: [PartitionSpec(spec_id=0)]
Current schema ID: 0


---
## 6. Ibis Multi-Engine Access

Access the same data through DuckDB, Polars, or PySpark Ibis backends.
DuckDB and Polars are JVM-free; PySpark requires a JVM.

In [ ]:
# DuckDB via Ibis (no JVM)
duck = conn.ibis_duckdb()
print("--- DuckDB Ibis backend ---")
result = duck.sql("SELECT name, age FROM lakekeeper.default.demo_users WHERE age >= 30")
result.execute()

--- DuckDB Ibis backend ---


,name,age
0,Alice,30
1,Charlie,35


In [ ]:
# Polars via Ibis (no JVM) — table is loaded via PyIceberg and registered
polars_backend = conn.ibis_polars("default", "demo_users")
print("--- Polars Ibis backend ---")
t = polars_backend.table("default.demo_users")
t.filter(t.age < 30).execute()

--- Polars Ibis backend ---


,id,name,age
0,2,Bob,25


In [ ]:
# Clean up the LakehouseConnection
conn.close()

---
## 7. PySpark Direct Access

For full Spark ecosystem access, use the catalog-specific builders.
These configure Spark with the correct JARs, extensions, and catalog settings.

In [ ]:
from poor_man_lakehouse import CatalogType, get_spark_builder

builder = get_spark_builder(CatalogType.LAKEKEEPER)
spark = builder.get_spark_session()
print(f"Spark version: {spark.version}")
print(f"Catalog: {spark.catalog.currentCatalog()}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 00:13:13 WARN Utils: Your hostname, MacBookPro.homenet.telecomitalia.it, resolves to a loopback address: 127.0.0.1; using 192.168.1.81 instead (on interface en0)
26/04/02 00:13:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/graziano/apache-spark/4.0.1/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/graziano/.ivy2.5.2/cache
The jars for the packages stored in: /Users/graziano/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.postgresql#postgresql added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.13 added as a dependency
:: resolving dependencie

Spark version: 4.0.1
Catalog: lakekeeper


In [ ]:
# Query the table we created earlier via DuckDB
spark.sql("SELECT * FROM lakekeeper.default.demo_users ORDER BY id").show()

# Insert more data via Spark
spark.sql("""
    INSERT INTO lakekeeper.default.demo_users
    VALUES (4, 'Diana', 28), (5, 'Eve', 32)
""")
spark.sql("SELECT * FROM lakekeeper.default.demo_users ORDER BY id").show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 25|
|  3|Charlie| 35|
+---+-------+---+

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 25|
|  3|Charlie| 35|
|  4|  Diana| 28|
|  5|    Eve| 32|
+---+-------+---+



In [ ]:
spark.stop()

---
## 8. Sail — Rust-Powered Spark Connect Engine (No JVM)

[Sail](https://docs.lakesail.com/sail/latest/) is a Rust-based compute engine that implements
the Spark Connect protocol. It provides a PySpark-compatible API **without a JVM**, with native
support for Parquet, Delta Lake, Iceberg (local), and S3/MinIO.

Sail reads S3 credentials from environment variables (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`,
`AWS_ENDPOINT_URL`, `AWS_ALLOW_HTTP`), not from Spark config.

> **Note:** Sail does not support REST catalogs (Lakekeeper/Nessie) yet. It works with
> local file-based table formats and S3 storage directly.

In [ ]:
from pysail.spark import SparkConnectServer
from pyspark.sql import SparkSession

# Start Sail server (Rust engine, no JVM needed)
server = SparkConnectServer()
server.start()
addr = server.listening_address
print(f"Sail server listening on {addr[0]}:{addr[1]}")

# Connect PySpark via Spark Connect protocol
sail_spark = SparkSession.builder.remote(f"sc://{addr[0]}:{addr[1]}").getOrCreate()
print(f"Spark version (via Sail): {sail_spark.version}")

Sail server listening on 127.0.0.1:51137
Spark version (via Sail): 4.1.1


In [ ]:
# Basic DataFrame operations — same PySpark API, Rust engine under the hood
df = sail_spark.createDataFrame(
    [(1, "Alice", 30), (2, "Bob", 25), (3, "Charlie", 35), (4, "Diana", 28)],
    ["id", "name", "age"],
)
df.show()

# SQL analytics
df.createOrReplaceTempView("sail_users")
sail_spark.sql("SELECT name, age, RANK() OVER (ORDER BY age DESC) as rank FROM sail_users").show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 25|
|  3|Charlie| 35|
|  4|  Diana| 28|
+---+-------+---+

+-------+---+----+
|   name|age|rank|
+-------+---+----+
|Charlie| 35|   1|
|  Alice| 30|   2|
|  Diana| 28|   3|
|    Bob| 25|   4|
+-------+---+----+



In [ ]:
# Delta Lake on S3/MinIO (requires AWS_* env vars set before starting the kernel)
try:
    df.write.format("delta").mode("overwrite").save("s3://warehouse/sail_demo/delta")
    sail_spark.read.format("delta").load("s3://warehouse/sail_demo/delta").show()
    print("Sail + Delta on S3: OK")
except Exception as e:
    print(f"Sail + Delta on S3: {e}")
    print("(Set AWS_* env vars before starting the kernel for S3 access)")

Sail + Delta on S3: URL scheme is not allowed
(Set AWS_* env vars before starting the kernel for S3 access)


In [ ]:
# Local Iceberg (file-based, no catalog server needed)
import tempfile

tmp = tempfile.mkdtemp()
df.write.format("iceberg").mode("overwrite").save(f"file://{tmp}/sail_iceberg/")
sail_spark.read.format("iceberg").load(f"file://{tmp}/sail_iceberg/").show()
print("Sail + Iceberg (local): OK")

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 25|
|  3|Charlie| 35|
|  4|  Diana| 28|
+---+-------+---+

Sail + Iceberg (local): OK


In [ ]:
sail_spark.stop()
server.stop()
print("Sail server stopped")

Sail server stopped


---
## Summary

| Entry Point | JVM Required | Read | Write | Best For |
|-------------|:---:|:---:|:---:|----------|
| `get_catalog()` | No | PyIceberg API | No | Raw catalog access, advanced metadata |
| `LakehouseConnection` | PySpark only | Polars, Arrow, DuckDB, Ibis | DuckDB | Most use cases (browsing, scanning, SQL, writes) |
| `SparkBuilder` | Yes | PySpark | PySpark | Full Spark ecosystem, complex ETL |
| `Sail` | No (Rust) | PySpark API | Delta, Iceberg, Parquet | JVM-free Spark alternative |

### Supported Catalogs

| Catalog | `get_catalog()` | `LakehouseConnection` | `SparkBuilder` |
|---------|:---:|:---:|:---:|
| **Nessie** | REST | All features | `NessieCatalogSparkBuilder` |
| **Lakekeeper** | REST | All features | `LakekeeperCatalogSparkBuilder` |
| **PostgreSQL** | SQL | Browsing + scans | `PostgresCatalogSparkBuilder` |
| **AWS Glue** | Glue | All features | `GlueCatalogSparkBuilder` |